# Home Credit Default Risk (End-to-End Pipeline)

Notebook ini menyusun alur end-to-end untuk Home Credit Default Risk.

Metodologi mengacu pada paper:
Zhang, X. et al. (2025). *Data-Driven Loan Default Prediction: A Machine Learning Approach for Enhancing Business Process Management*. Systems, 13(7), 581. https://doi.org/10.3390/systems13070581

Model yang dicoba: **Logistic Regression (wajib)**, Random Forest,
Gradient Boosting, XGBoost, LightGBM, + Voting Ensemble.
Hyperparameter tuning belum dilakukan di sini dan akan dikerjakan di tahap lanjut.

## 0. Import Library

In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # ganti ke backend interaktif kalau ingin plot tampil langsung
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFECV, SelectFromModel
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve
)

from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## Konfigurasi

In [ ]:
# PATHS
BASE_DIR = os.getcwd()  # notebook pakai working directory sebagai acuan
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")

for d in [OUTPUT_DIR, FIG_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

RAW_FILES = {
    "application_train": "application_train.csv",
    "application_test": "application_test.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "pos_cash_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "installments_payments": "installments_payments.csv",
}

MASTER_DATASET_PATH = os.path.join(OUTPUT_DIR, "master_dataset.parquet")
MASTER_DATASET_CSV_PATH = os.path.join(OUTPUT_DIR, "master_dataset.csv")

# CONSTANTS
TARGET_COL = "TARGET"
ID_COL = "SK_ID_CURR"
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_CV_FOLDS = 5

# Dev mode: jika True, hanya baca sebagian baris supaya pipeline cepat dicoba
DEV_MODE = False
DEV_NROWS_MAIN = 50_000
DEV_NROWS_AUX = 200_000

# Kolom dengan missing value di atas threshold ini akan dibuang
MISSING_THRESHOLD = 0.90

# Batas fitur setelah SelectFromModel
N_FEATURES_SELECT_FROM_MODEL = 100

# RFECV mahal, jadi default dijalankan pada subsample
RUN_RFECV = True
RFECV_SUBSAMPLE_N = 20_000
RFECV_STEP = 5
RFECV_CV = 3
RFECV_MIN_FEATURES = 20

# Jumlah fitur final yang dipakai model
N_FINAL_FEATURES = 40

In [3]:
# Objek `config` (namespace) supaya fungsi-fungsi yang menerima parameter
# `config` (mis. run_feature_selection(X, y, config)) tetap bisa dipanggil
# persis seperti pada file .py aslinya, tanpa mengubah isi fungsi tsb.
from types import SimpleNamespace

config = SimpleNamespace(
    BASE_DIR=BASE_DIR, DATA_DIR=DATA_DIR, OUTPUT_DIR=OUTPUT_DIR,
    FIG_DIR=FIG_DIR, MODEL_DIR=MODEL_DIR, RAW_FILES=RAW_FILES,
    MASTER_DATASET_PATH=MASTER_DATASET_PATH, MASTER_DATASET_CSV_PATH=MASTER_DATASET_CSV_PATH,
    TARGET_COL=TARGET_COL, ID_COL=ID_COL, RANDOM_STATE=RANDOM_STATE,
    TEST_SIZE=TEST_SIZE, N_CV_FOLDS=N_CV_FOLDS, DEV_MODE=DEV_MODE,
    DEV_NROWS_MAIN=DEV_NROWS_MAIN, DEV_NROWS_AUX=DEV_NROWS_AUX,
    MISSING_THRESHOLD=MISSING_THRESHOLD,
    N_FEATURES_SELECT_FROM_MODEL=N_FEATURES_SELECT_FROM_MODEL,
    RUN_RFECV=RUN_RFECV, RFECV_SUBSAMPLE_N=RFECV_SUBSAMPLE_N,
    RFECV_STEP=RFECV_STEP, RFECV_CV=RFECV_CV, RFECV_MIN_FEATURES=RFECV_MIN_FEATURES,
    N_FINAL_FEATURES=N_FINAL_FEATURES,
)

## 1. `data_loading.py` — Load ke-7 tabel mentah

In [ ]:
"""
data_loading.py
Membaca tujuh tabel mentah Home Credit dari DATA_DIR.

Semua fungsi mengembalikan pandas DataFrame. Kolom SK_ID_CURR,
SK_ID_PREV, dan SK_ID_BUREAU dipertahankan dalam bentuk yang hemat
memori jika memungkinkan.
"""


def _check_file(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File '{name}' tidak ditemukan di '{path}'.\n"
        )


def _read_csv(key: str, nrows: int = None) -> pd.DataFrame:
    path = os.path.join(DATA_DIR, RAW_FILES[key])
    _check_file(path, RAW_FILES[key])
    df = pd.read_csv(path, nrows=nrows)
    return df


def load_application_train() -> pd.DataFrame:
    nrows = DEV_NROWS_MAIN if DEV_MODE else None
    df = _read_csv("application_train", nrows=nrows)
    print(f"[load] application_train: {df.shape}")
    return df


def load_application_test() -> pd.DataFrame:
    df = _read_csv("application_test")
    print(f"[load] application_test: {df.shape}")
    return df


def load_bureau() -> pd.DataFrame:
    nrows = DEV_NROWS_AUX if DEV_MODE else None
    df = _read_csv("bureau", nrows=nrows)
    print(f"[load] bureau: {df.shape}")
    return df


def load_bureau_balance() -> pd.DataFrame:
    nrows = DEV_NROWS_AUX if DEV_MODE else None
    df = _read_csv("bureau_balance", nrows=nrows)
    print(f"[load] bureau_balance: {df.shape}")
    return df


def load_previous_application() -> pd.DataFrame:
    nrows = DEV_NROWS_AUX if DEV_MODE else None
    df = _read_csv("previous_application", nrows=nrows)
    print(f"[load] previous_application: {df.shape}")
    return df


def load_pos_cash_balance() -> pd.DataFrame:
    nrows = DEV_NROWS_AUX if DEV_MODE else None
    df = _read_csv("pos_cash_balance", nrows=nrows)
    print(f"[load] POS_CASH_balance: {df.shape}")
    return df


def load_credit_card_balance() -> pd.DataFrame:
    nrows = DEV_NROWS_AUX if DEV_MODE else None
    df = _read_csv("credit_card_balance", nrows=nrows)
    print(f"[load] credit_card_balance: {df.shape}")
    return df


def load_installments_payments() -> pd.DataFrame:
    nrows = DEV_NROWS_AUX if DEV_MODE else None
    df = _read_csv("installments_payments", nrows=nrows)
    print(f"[load] installments_payments: {df.shape}")
    return df


def load_all_raw_tables() -> dict:
    """Load semua tabel mentah sekaligus dan kembalikan sebagai dict."""
    tables = {
        "application_train": load_application_train(),
        "application_test": load_application_test(),
        "bureau": load_bureau(),
        "bureau_balance": load_bureau_balance(),
        "previous_application": load_previous_application(),
        "pos_cash_balance": load_pos_cash_balance(),
        "credit_card_balance": load_credit_card_balance(),
        "installments_payments": load_installments_payments(),
    }
    return tables

## 2. Join jadi satu dataset besar
Tabel anak (bureau, previous_application, POS_CASH_balance,
credit_card_balance, installments_payments) diagregasi dulu ke level
SK_ID_CURR, lalu digabung ke application_train / application_test.

In [ ]:
"""
Mengagregasi tabel-tabel anak seperti bureau, previous_application,
POS_CASH, credit_card, dan installments ke level SK_ID_CURR, lalu
menggabungkannya ke application_train dan application_test.

Karena satu SK_ID_CURR bisa punya banyak baris di tabel anak, semua
riwayat perlu diringkas dulu sebelum bisa dipakai sebagai fitur 1:1.
"""


def _agg_numeric(df: pd.DataFrame, group_col: str, prefix: str,
                  stats=("min", "max", "mean", "sum", "std")) -> pd.DataFrame:
    """Agregasi kolom numerik per group_col."""
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c != group_col]
    if not num_cols:
        return df[[group_col]].drop_duplicates().set_index(group_col)
    agg = df.groupby(group_col)[num_cols].agg(list(stats))
    agg.columns = [f"{prefix}_{col}_{stat.upper()}" for col, stat in agg.columns]
    return agg


def _agg_categorical(df: pd.DataFrame, group_col: str, prefix: str) -> pd.DataFrame:
    """One-hot encode kategori lalu ambil mean dan sum per group."""
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    cat_cols = [c for c in cat_cols if c != group_col]
    if not cat_cols:
        return df[[group_col]].drop_duplicates().set_index(group_col)
    dummies = pd.get_dummies(df[cat_cols], dummy_na=True)
    dummies[group_col] = df[group_col].values
    agg = dummies.groupby(group_col).agg(["mean", "sum"])
    agg.columns = [f"{prefix}_{col}_{stat.upper()}" for col, stat in agg.columns]
    return agg


def _count_per_id(df: pd.DataFrame, group_col: str, prefix: str,
                   count_name: str = "COUNT") -> pd.DataFrame:
    counts = df.groupby(group_col).size().to_frame(f"{prefix}_{count_name}")
    return counts


# bureau_balance -> agregasi per SK_ID_BUREAU, lalu digabung ke bureau
def aggregate_bureau_balance(bureau_balance: pd.DataFrame) -> pd.DataFrame:
    num_agg = _agg_numeric(bureau_balance, "SK_ID_BUREAU", "BB",
                            stats=("min", "max", "mean"))
    cat_agg = _agg_categorical(bureau_balance, "SK_ID_BUREAU", "BB")
    cnt = _count_per_id(bureau_balance, "SK_ID_BUREAU", "BB")
    out = num_agg.join(cat_agg, how="outer").join(cnt, how="outer")
    return out.reset_index()


# bureau (+ bureau_balance) -> agregasi per SK_ID_CURR
def aggregate_bureau(bureau: pd.DataFrame, bureau_balance_agg: pd.DataFrame) -> pd.DataFrame:
    bureau = bureau.merge(bureau_balance_agg, on="SK_ID_BUREAU", how="left")

    num_agg = _agg_numeric(bureau, "SK_ID_CURR", "BUREAU")
    cat_agg = _agg_categorical(bureau, "SK_ID_CURR", "BUREAU")
    cnt = _count_per_id(bureau, "SK_ID_CURR", "BUREAU")

    out = num_agg.join(cat_agg, how="outer").join(cnt, how="outer")
    return out.reset_index()


# previous_application -> agregasi per SK_ID_CURR
def aggregate_previous_application(prev: pd.DataFrame) -> pd.DataFrame:
    # DAYS_* bernilai 365243 berarti "tidak ada" -> jadikan NaN
    days_cols = [c for c in prev.columns if c.startswith("DAYS_")]
    for c in days_cols:
        prev[c] = prev[c].replace(365243, np.nan)

    num_agg = _agg_numeric(prev, "SK_ID_CURR", "PREV")
    cat_agg = _agg_categorical(prev, "SK_ID_CURR", "PREV")
    cnt = _count_per_id(prev, "SK_ID_CURR", "PREV")

    out = num_agg.join(cat_agg, how="outer").join(cnt, how="outer")
    return out.reset_index()


# POS_CASH_balance -> agregasi per SK_ID_CURR
def aggregate_pos_cash(pos: pd.DataFrame) -> pd.DataFrame:
    num_agg = _agg_numeric(pos, "SK_ID_CURR", "POS")
    cat_agg = _agg_categorical(pos, "SK_ID_CURR", "POS")
    cnt = _count_per_id(pos, "SK_ID_CURR", "POS")

    out = num_agg.join(cat_agg, how="outer").join(cnt, how="outer")
    return out.reset_index()


# credit_card_balance -> agregasi per SK_ID_CURR
def aggregate_credit_card(cc: pd.DataFrame) -> pd.DataFrame:
    num_agg = _agg_numeric(cc, "SK_ID_CURR", "CC")
    cat_agg = _agg_categorical(cc, "SK_ID_CURR", "CC")
    cnt = _count_per_id(cc, "SK_ID_CURR", "CC")

    out = num_agg.join(cat_agg, how="outer").join(cnt, how="outer")
    return out.reset_index()


# installments_payments -> agregasi per SK_ID_CURR
def aggregate_installments(ins: pd.DataFrame) -> pd.DataFrame:
    # Selisih dan rasio bayar yang paling sering dipakai sebagai fitur ringkas
    ins = ins.copy()
    ins["PAYMENT_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]
    ins["PAYMENT_RATIO"] = ins["AMT_PAYMENT"] / (ins["AMT_INSTALMENT"] + 1e-9)
    ins["DAYS_LATE"] = ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]

    num_agg = _agg_numeric(ins, "SK_ID_CURR", "INSTAL")
    cnt = _count_per_id(ins, "SK_ID_CURR", "INSTAL")

    out = num_agg.join(cnt, how="outer")
    return out.reset_index()


# Gabungkan semuanya menjadi satu dataset besar
def build_master_dataset(tables: dict) -> tuple:
    """
    tables: dict berisi key -> DataFrame, hasil dari data_loading.load_all_raw_tables()
    Mengembalikan (df_train, df_test) yang sudah digabung dengan seluruh
    fitur agregat dari tabel anak.
    """
    app_train = tables["application_train"].copy()
    app_test = tables["application_test"].copy()

    print("[join] Menggabungkan application_train & application_test agar "
          "encoding & agregasi konsisten...")
    app_train["IS_TRAIN"] = 1
    app_test["IS_TRAIN"] = 0
    if "TARGET" not in app_test.columns:
        app_test["TARGET"] = np.nan
    app = pd.concat([app_train, app_test], axis=0, ignore_index=True, sort=False)
    print(f"[join] application gabungan: {app.shape}")

    print("[join] Agregasi bureau_balance -> bureau ...")
    bb_agg = aggregate_bureau_balance(tables["bureau_balance"])
    bureau_agg = aggregate_bureau(tables["bureau"], bb_agg)
    print(f"[join] bureau_agg: {bureau_agg.shape}")

    print("[join] Agregasi previous_application ...")
    prev_agg = aggregate_previous_application(tables["previous_application"])
    print(f"[join] prev_agg: {prev_agg.shape}")

    print("[join] Agregasi POS_CASH_balance ...")
    pos_agg = aggregate_pos_cash(tables["pos_cash_balance"])
    print(f"[join] pos_agg: {pos_agg.shape}")

    print("[join] Agregasi credit_card_balance ...")
    cc_agg = aggregate_credit_card(tables["credit_card_balance"])
    print(f"[join] cc_agg: {cc_agg.shape}")

    print("[join] Agregasi installments_payments ...")
    ins_agg = aggregate_installments(tables["installments_payments"])
    print(f"[join] ins_agg: {ins_agg.shape}")

    master = app
    for name, agg_df in [
        ("bureau", bureau_agg),
        ("previous_application", prev_agg),
        ("pos_cash", pos_agg),
        ("credit_card", cc_agg),
        ("installments", ins_agg),
    ]:
        before_cols = master.shape[1]
        master = master.merge(agg_df, on="SK_ID_CURR", how="left")
        print(f"[join] merge {name}: {before_cols} -> {master.shape[1]} kolom")

    print(f"[join] Dataset akhir (master): {master.shape}")

    df_train = master[master["IS_TRAIN"] == 1].drop(columns=["IS_TRAIN"]).reset_index(drop=True)
    df_test = master[master["IS_TRAIN"] == 0].drop(columns=["IS_TRAIN", "TARGET"]).reset_index(drop=True)

    return df_train, df_test

## 3. Data Wrangling (tahap 1 — sebelum EDA)
- Perbaikan anomali, misalnya `DAYS_EMPLOYED = 365243`
- Feature engineering rasio finansial seperti credit/income,
  annuity/income, dan employed/age
- Buang kolom dengan missing value di atas threshold

Encoding kategorikal sengaja ditunda sampai setelah EDA supaya label
aslinya masih mudah dibaca di plot.

In [6]:
"""
wrangling.py
Data wrangling untuk dataset gabungan (master dataset):
- Perbaikan anomali (DAYS_EMPLOYED = 365243, dsb.)
- Feature engineering rasio finansial (mengikuti semangat paper referensi:
  debt-to-income ratio, age-to-experience ratio, dsb., diadaptasi ke
  variabel yang tersedia di dataset Home Credit)
- Encoding kategorikal (LabelEncoder utk biner, One-Hot utk multi-kategori
  -> sesuai paper utk model tree, tapi one-hot dipakai juga supaya
  Logistic Regression bisa memakai data yang sama)
- Penanganan missing value
"""


def fix_anomalies(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # DAYS_EMPLOYED = 365243 adalah kode "tidak bekerja / pensiunan" -> jadikan NaN + flag
    if "DAYS_EMPLOYED" in df.columns:
        df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
        df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

    # DAYS_* lain seharusnya negatif (hari relatif terhadap aplikasi); jadikan positif untuk interpretasi
    for c in ["DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION",
              "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE"]:
        if c in df.columns:
            df[c] = df[c].abs()

    return df


def engineer_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fitur turunan terinspirasi dari paper:
      - debt_to_income analog     -> CREDIT_INCOME_RATIO, ANNUITY_INCOME_RATIO
      - age_to_experience analog  -> DAYS_EMPLOYED_TO_BIRTH_RATIO (usia vs lama bekerja)
      - credit_term (tenor kredit tersirat)
    """
    df = df.copy()
    eps = 1e-9

    if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / (df["AMT_INCOME_TOTAL"] + eps)

    if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + eps)

    if {"AMT_ANNUITY", "AMT_CREDIT"}.issubset(df.columns):
        df["CREDIT_TERM"] = df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + eps)

    if {"AMT_GOODS_PRICE", "AMT_CREDIT"}.issubset(df.columns):
        df["GOODS_PRICE_CREDIT_RATIO"] = df["AMT_GOODS_PRICE"] / (df["AMT_CREDIT"] + eps)

    if {"DAYS_EMPLOYED", "DAYS_BIRTH"}.issubset(df.columns):
        # analog "age-to-experience ratio" di paper: proporsi hidup yang dihabiskan bekerja
        df["EMPLOYED_TO_BIRTH_RATIO"] = df["DAYS_EMPLOYED"] / (df["DAYS_BIRTH"] + eps)

    if {"AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS"}.issubset(df.columns):
        df["INCOME_PER_FAMILY_MEMBER"] = df["AMT_INCOME_TOTAL"] / (df["CNT_FAM_MEMBERS"].replace(0, np.nan))

    if {"DAYS_BIRTH"}.issubset(df.columns):
        df["AGE_YEARS"] = df["DAYS_BIRTH"] / 365.25

    if {"DAYS_EMPLOYED"}.issubset(df.columns):
        df["EMPLOYED_YEARS"] = df["DAYS_EMPLOYED"] / 365.25

    # Ratio dari agregat kredit biro (bureau) jika tersedia
    if {"BUREAU_AMT_CREDIT_SUM_SUM", "AMT_INCOME_TOTAL"}.issubset(df.columns):
        df["BUREAU_CREDIT_TO_INCOME"] = df["BUREAU_AMT_CREDIT_SUM_SUM"] / (df["AMT_INCOME_TOTAL"] + eps)

    # Replace inf yang mungkin muncul dari pembagian
    df = df.replace([np.inf, -np.inf], np.nan)
    return df


def drop_high_missing(df: pd.DataFrame, threshold: float, protect_cols=("SK_ID_CURR", "TARGET")) -> pd.DataFrame:
    missing_frac = df.isna().mean()
    to_drop = [c for c in missing_frac[missing_frac > threshold].index if c not in protect_cols]
    print(f"[wrangling] Membuang {len(to_drop)} kolom dgn missing > {threshold*100:.0f}%")
    return df.drop(columns=to_drop)


def encode_categoricals(df_train: pd.DataFrame, df_test: pd.DataFrame):
    """
    - Kolom biner (2 kategori unik, termasuk NaN dihitung terpisah) -> LabelEncoder
      (sesuai paper: person_gender dsb. di-LabelEncode)
    - Kolom multi-kategori -> One-Hot Encoding
      (dipakai tambahan supaya Logistic Regression bisa memakai fitur yang sama;
       model tree tetap bisa memakai representasi ini juga)
    df_train harus punya kolom TARGET, df_test tidak.
    """
    df_train = df_train.copy()
    df_test = df_test.copy()

    cat_cols = df_train.select_dtypes(include=["object", "category"]).columns.tolist()

    binary_cols, multi_cols = [], []
    for c in cat_cols:
        n_unique = df_train[c].nunique(dropna=True)
        if n_unique <= 2:
            binary_cols.append(c)
        else:
            multi_cols.append(c)

    # Label encode kolom biner
    for c in binary_cols:
        le = LabelEncoder()
        combined = pd.concat([df_train[c], df_test[c]], axis=0).astype(str)
        le.fit(combined)
        df_train[c] = le.transform(df_train[c].astype(str))
        df_test[c] = le.transform(df_test[c].astype(str))

    # One-hot kolom multi-kategori, align kolom train/test
    if multi_cols:
        train_dummies = pd.get_dummies(df_train[multi_cols], dummy_na=True, prefix=multi_cols)
        test_dummies = pd.get_dummies(df_test[multi_cols], dummy_na=True, prefix=multi_cols)
        train_dummies, test_dummies = train_dummies.align(test_dummies, join="left", axis=1, fill_value=0)

        df_train = pd.concat([df_train.drop(columns=multi_cols), train_dummies], axis=1)
        df_test = pd.concat([df_test.drop(columns=multi_cols), test_dummies], axis=1)

    print(f"[wrangling] Label-encoded (biner): {len(binary_cols)} kolom")
    print(f"[wrangling] One-hot encoded (multi-kategori): {len(multi_cols)} kolom")

    return df_train, df_test


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Beberapa library (LightGBM/XGBoost) tidak suka karakter spesial di nama kolom."""
    df = df.copy()
    df.columns = [
        "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in str(c))
        for c in df.columns
    ]
    return df


def wrangle_pre_eda(df_train: pd.DataFrame, df_test: pd.DataFrame, missing_threshold: float):
    """Tahap wrangling SEBELUM EDA: perbaikan anomali, feature engineering rasio,
    dan pembuangan kolom high-missing. Kolom kategorikal masih dalam bentuk
    string aslinya (mis. 'M'/'F', 'Higher education') supaya EDA tetap mudah
    dibaca / diinterpretasi."""
    df_train = fix_anomalies(df_train)
    df_test = fix_anomalies(df_test)

    df_train = engineer_ratio_features(df_train)
    df_test = engineer_ratio_features(df_test)

    df_train = drop_high_missing(df_train, missing_threshold)
    df_test = df_test[[c for c in df_test.columns if c in df_train.columns or c == "SK_ID_CURR"]]

    return df_train, df_test


def wrangle_encode(df_train: pd.DataFrame, df_test: pd.DataFrame):
    """Tahap wrangling SETELAH EDA: encoding kategorikal (LabelEncoder utk
    biner, One-Hot utk multi-kategori) + pembersihan nama kolom, siap untuk
    feature selection & modeling."""
    df_train, df_test = encode_categoricals(df_train, df_test)
    df_train = clean_column_names(df_train)
    df_test = clean_column_names(df_test)
    return df_train, df_test


def wrangle(df_train: pd.DataFrame, df_test: pd.DataFrame, missing_threshold: float):
    """Wrapper lengkap (pre-EDA + encode) untuk kompatibilitas / dipakai
    di luar konteks EDA."""
    df_train, df_test = wrangle_pre_eda(df_train, df_test, missing_threshold)
    df_train, df_test = wrangle_encode(df_train, df_test)
    return df_train, df_test

## 4. EDA (Exploratory Data Analysis)
Lihat pola dasar data sebelum masuk ke encoding dan modeling.

In [ ]:
"""
eda.py
Exploratory Data Analysis untuk dataset Home Credit.
Menghasilkan plot-plot serupa dengan paper referensi (distribusi usia,
income, credit amount, interest-proxy, default rate per kategori, dll.)
dan menyimpannya sebagai file gambar.
"""


def _save(fig, fig_dir, filename):
    path = os.path.join(fig_dir, filename)
    fig.savefig(path, bbox_inches="tight", dpi=120)
    plt.close(fig)
    print(f"[eda] saved: {path}")


def target_distribution(df: pd.DataFrame, fig_dir: str):
    fig, ax = plt.subplots(figsize=(5, 4))
    counts = df["TARGET"].value_counts().sort_index()
    sns.barplot(x=counts.index.astype(str), y=counts.values, ax=ax, hue=counts.index.astype(str), palette="viridis", legend=False)
    ax.set_xlabel("TARGET (0 = lunas, 1 = default)")
    ax.set_ylabel("Jumlah aplikasi")
    ax.set_title(f"Distribusi Target (imbalance ratio ~ 1:{counts[0]//max(counts[1],1)})")
    for i, v in enumerate(counts.values):
        ax.text(i, v, f"{v:,}\n({v/counts.sum()*100:.1f}%)", ha="center", va="bottom")
    _save(fig, fig_dir, "01_target_distribution.png")


def numeric_distribution(df: pd.DataFrame, col: str, fig_dir: str, filename: str,
                          title: str, clip_quantile=0.99):
    if col not in df.columns:
        return
    data = df[col].dropna()
    upper = data.quantile(clip_quantile)
    data = data[data <= upper]
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.histplot(data, kde=True, ax=ax, color="steelblue")
    ax.set_title(title)
    ax.set_xlabel(col)
    _save(fig, fig_dir, filename)


def income_vs_credit(df: pd.DataFrame, fig_dir: str):
    if not {"AMT_INCOME_TOTAL", "AMT_CREDIT"}.issubset(df.columns):
        return
    sample = df.sample(min(20000, len(df)), random_state=42)
    q_income = sample["AMT_INCOME_TOTAL"].quantile(0.99)
    sample = sample[sample["AMT_INCOME_TOTAL"] <= q_income]
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(data=sample, x="AMT_INCOME_TOTAL", y="AMT_CREDIT",
                     hue="TARGET", alpha=0.3, s=10, ax=ax, palette=["#2b8cbe", "#de2d26"])
    ax.set_title("Income vs Jumlah Kredit yang Diajukan")
    _save(fig, fig_dir, "04_income_vs_credit.png")


def default_rate_by_category(df: pd.DataFrame, col: str, fig_dir: str, filename: str, title: str):
    if col not in df.columns:
        return
    rate = df.groupby(col)["TARGET"].mean().sort_values(ascending=False)
    rate = rate[df.groupby(col).size() > 50]  # buang kategori dgn sample terlalu kecil
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=ax, hue=rate.index.astype(str), palette="rocket", legend=False)
    ax.set_ylabel("Default rate")
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    _save(fig, fig_dir, filename)


def missing_value_overview(df: pd.DataFrame, fig_dir: str, top_n=30):
    missing = df.isna().mean().sort_values(ascending=False)
    missing = missing[missing > 0].head(top_n)
    if missing.empty:
        return
    fig, ax = plt.subplots(figsize=(7, max(4, top_n * 0.25)))
    sns.barplot(x=missing.values * 100, y=missing.index, ax=ax, color="salmon")
    ax.set_xlabel("% missing")
    ax.set_title(f"Top {top_n} kolom dengan missing value terbanyak")
    _save(fig, fig_dir, "06_missing_values.png")


def correlation_heatmap(df: pd.DataFrame, fig_dir: str, top_n=20):
    num_df = df.select_dtypes(include=[np.number])
    corr_target = num_df.corr()["TARGET"].drop("TARGET").abs().sort_values(ascending=False)
    top_feats = corr_target.head(top_n).index.tolist()
    corr_matrix = num_df[top_feats + ["TARGET"]].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, cmap="coolwarm", center=0, ax=ax, annot=False)
    ax.set_title(f"Korelasi {top_n} fitur numerik teratas terhadap TARGET")
    _save(fig, fig_dir, "07_correlation_heatmap.png")
    return corr_target


def run_full_eda(df: pd.DataFrame, fig_dir: str):
    os.makedirs(fig_dir, exist_ok=True)
    print("[eda] Menjalankan EDA lengkap ...")

    target_distribution(df, fig_dir)

    # Usia dalam tahun (jika belum ada kolom AGE_YEARS, hitung sementara)
    tmp = df.copy()
    if "AGE_YEARS" not in tmp.columns and "DAYS_BIRTH" in tmp.columns:
        tmp["AGE_YEARS"] = tmp["DAYS_BIRTH"].abs() / 365.25
    numeric_distribution(tmp, "AGE_YEARS", fig_dir, "02_age_distribution.png",
                          "Distribusi Usia Peminjam")
    numeric_distribution(tmp, "AMT_INCOME_TOTAL", fig_dir, "03_income_distribution.png",
                          "Distribusi Income Peminjam (clip 99th pct)")
    income_vs_credit(tmp, fig_dir)
    numeric_distribution(tmp, "AMT_CREDIT", fig_dir, "05_credit_amount_distribution.png",
                          "Distribusi Jumlah Kredit yang Diajukan")

    missing_value_overview(df, fig_dir)

    corr_target = None
    try:
        corr_target = correlation_heatmap(df, fig_dir)
    except Exception as e:
        print(f"[eda] Lewati correlation heatmap: {e}")

    for col, fname, title in [
        ("CODE_GENDER", "08_default_rate_gender.png", "Default Rate berdasarkan Gender"),
        ("NAME_EDUCATION_TYPE", "09_default_rate_education.png", "Default Rate berdasarkan Pendidikan"),
        ("NAME_CONTRACT_TYPE", "10_default_rate_contract.png", "Default Rate berdasarkan Tipe Kontrak"),
    ]:
        default_rate_by_category(df, col, fig_dir, fname, title)

    print("[eda] Selesai. Semua figure disimpan di:", fig_dir)
    return corr_target

## 5. Feature Selection
Dua tahap sesuai paper: ranking fitur cepat dulu, lalu RFECV untuk
memilih kandidat akhir.

In [ ]:
"""
feature_selection.py
Seleksi fitur dua tahap mengikuti paper referensi:
  1) RFECV (Recursive Feature Elimination with Cross-Validation) - opsional,
     dijalankan pada subsample training untuk menjaga waktu komputasi wajar.
  2) SelectFromModel memakai feature_importances_ dari model tree
     (Random Forest / Gradient Boosting).

Untuk dataset yang digunakan pada notebook ini,
RFECV murni sangat mahal, jadi kita jalankan pada subsample lalu terapkan
hasilnya (fitur terpilih) ke keseluruhan data.
"""


def select_from_model_step(X: pd.DataFrame, y: pd.Series, n_features: int,
                            random_state: int = 42) -> list:
    """Tahap 1 (disederhanakan sbg langkah cepat): ranking fitur pakai
    Gradient Boosting importances, ambil top-N sebagai kandidat awal."""
    print(f"[feature_selection] SelectFromModel: melatih GradientBoosting utk ranking fitur ...")
    clf = GradientBoostingClassifier(
        n_estimators=100, max_depth=3, random_state=random_state, subsample=0.7
    )
    # subsample agar cepat
    if len(X) > 50_000:
        idx = X.sample(50_000, random_state=random_state).index
        clf.fit(X.loc[idx], y.loc[idx])
    else:
        clf.fit(X, y)

    importances = pd.Series(clf.feature_importances_, index=X.columns)
    top_feats = importances.sort_values(ascending=False).head(n_features).index.tolist()
    print(f"[feature_selection] {len(top_feats)} fitur teratas dipilih dari {X.shape[1]} fitur awal.")
    return top_feats


def rfecv_step(X: pd.DataFrame, y: pd.Series, subsample_n: int, step: int,
               cv: int, min_features: int, random_state: int = 42) -> list:
    """Tahap 2: RFECV pada subsample data, memakai Random Forest sbg estimator."""
    print(f"[feature_selection] RFECV: menjalankan pada subsample n={min(subsample_n, len(X))} ...")
    if len(X) > subsample_n:
        idx = X.sample(subsample_n, random_state=random_state, weights=None).index
        X_sub, y_sub = X.loc[idx], y.loc[idx]
    else:
        X_sub, y_sub = X, y

    estimator = RandomForestClassifier(
        n_estimators=100, max_depth=8, n_jobs=-1, random_state=random_state,
        class_weight="balanced"
    )
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    selector = RFECV(
        estimator, step=step, cv=skf, scoring="f1",
        min_features_to_select=min_features, n_jobs=-1
    )
    selector.fit(X_sub, y_sub)
    selected = X.columns[selector.support_].tolist()
    print(f"[feature_selection] RFECV memilih {len(selected)} fitur "
          f"(dari {len(X.columns)} kandidat).")
    return selected


def run_feature_selection(X: pd.DataFrame, y: pd.Series, config) -> list:
    """Pipeline lengkap: SelectFromModel dulu (cepat, kurangi dimensi kasar),
    lalu RFECV opsional pada hasilnya untuk fine-tune."""
    candidate_feats = select_from_model_step(
        X, y, n_features=config.N_FEATURES_SELECT_FROM_MODEL,
        random_state=config.RANDOM_STATE,
    )

    if config.RUN_RFECV:
        final_feats = rfecv_step(
            X[candidate_feats], y,
            subsample_n=config.RFECV_SUBSAMPLE_N,
            step=config.RFECV_STEP,
            cv=config.RFECV_CV,
            min_features=config.RFECV_MIN_FEATURES,
            random_state=config.RANDOM_STATE,
        )
    else:
        final_feats = candidate_feats[:config.N_FINAL_FEATURES]

    if len(final_feats) > config.N_FINAL_FEATURES:
        # potong ke N_FINAL_FEATURES berdasar urutan importance awal
        rank = {f: i for i, f in enumerate(candidate_feats)}
        final_feats = sorted(final_feats, key=lambda f: rank.get(f, 999))[:config.N_FINAL_FEATURES]

    print(f"[feature_selection] Fitur final untuk modeling: {len(final_feats)}")
    return final_feats

## 6. Modeling
Model dilatih pada data hasil SMOTE di atas train split, tanpa menyentuh
test set. Semua model juga tetap memakai `class_weight="balanced"`
sebagai lapisan tambahan untuk menangani imbalance.

Model yang dicoba: **Logistic Regression**, Random Forest,
Gradient Boosting, XGBoost, LightGBM, + Voting Ensemble.

In [ ]:
"""
modeling.py
Definisi & training model, mengikuti paper referensi:
  - Logistic Regression 
  - Random Forest
  - Gradient Boosting
  - XGBoost
  - LightGBM
  - Voting Classifier (ensemble dari model-model terbaik, sesuai paper Sec. 3.6)

Class imbalance ditangani dengan dua cara (sesuai paper):
  1) class_weight='balanced' (built-in tiap model)
  2) SMOTE pada data training saja (tidak pernah pada data test!)
"""


def apply_smote(X_train: pd.DataFrame, y_train: pd.Series, random_state: int = 42):
    print(f"[modeling] SMOTE sebelum: {y_train.value_counts().to_dict()}")
    sm = SMOTE(random_state=random_state)
    X_res, y_res = sm.fit_resample(X_train, y_train)
    print(f"[modeling] SMOTE sesudah: {pd.Series(y_res).value_counts().to_dict()}")
    return X_res, y_res


def build_models(random_state: int = 42) -> dict:
    """Definisi model sesuai paper + Logistic Regression wajib.
    Hyperparameter memakai nilai default yang wajar (hp tuning menyusul)."""

    log_reg = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=random_state
        )),
    ])

    rf = RandomForestClassifier(
        n_estimators=300, max_depth=10, max_features="sqrt",
        min_samples_split=10, class_weight="balanced",
        n_jobs=-1, random_state=random_state,
    )

    gb = GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, random_state=random_state,
    )

    xgb_clf = xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=random_state,
        n_jobs=-1, tree_method="hist",
    )

    lgb_clf = lgb.LGBMClassifier(
        n_estimators=300, max_depth=-1, num_leaves=31,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        class_weight="balanced", random_state=random_state, n_jobs=-1, verbosity=-1,
    )

    models = {
        "LogisticRegression": log_reg,
        "RandomForest": rf,
        "GradientBoosting": gb,
        "XGBoost": xgb_clf,
        "LightGBM": lgb_clf,
    }
    return models


def build_voting_ensemble(fitted_models: dict, use_keys=("XGBoost", "GradientBoosting", "RandomForest")):
    """Voting classifier (soft voting) dari model-model terbaik, sesuai Sec. 3.6 paper."""
    estimators = [(k, fitted_models[k]) for k in use_keys if k in fitted_models]
    voting = VotingClassifier(estimators=estimators, voting="soft", n_jobs=-1)
    return voting


def train_all_models(X_train, y_train, X_train_smote=None, y_train_smote=None,
                      random_state: int = 42) -> dict:
    """
    Melatih semua model. Model linear/tree pakai class_weight='balanced'
    pada X_train asli (tidak di-SMOTE) KECUALI jika X_train_smote diberikan,
    dalam hal ini model dilatih pada data hasil SMOTE.
    """
    models = build_models(random_state=random_state)
    fitted = {}
    timings = {}

    X_fit = X_train_smote if X_train_smote is not None else X_train
    y_fit = y_train_smote if y_train_smote is not None else y_train

    for name, model in models.items():
        print(f"[modeling] Melatih {name} ...")
        t0 = time.time()
        model.fit(X_fit, y_fit)
        elapsed = time.time() - t0
        timings[name] = elapsed
        fitted[name] = model
        print(f"[modeling] {name} selesai dalam {elapsed:.1f} detik")

    # Voting ensemble dari model tree yang sudah fit (refit ulang dgn VotingClassifier
    # perlu estimator belum fit-nya sendiri; sklearn VotingClassifier akan fit ulang)
    print("[modeling] Melatih Voting Ensemble (XGBoost + GradientBoosting + RandomForest) ...")
    t0 = time.time()
    voting_models = build_models(random_state=random_state)
    voting = build_voting_ensemble(voting_models)
    voting.fit(X_fit, y_fit)
    timings["VotingEnsemble"] = time.time() - t0
    fitted["VotingEnsemble"] = voting
    print(f"[modeling] VotingEnsemble selesai dalam {timings['VotingEnsemble']:.1f} detik")

    return fitted, timings

## 7. `evaluation.py` — Metrik, Confusion Matrix, ROC/PR Curve, Feature Importance, CV
*(isi persis sama dengan `evaluation.py`)*

In [ ]:
def _save(fig, fig_dir, filename):
    path = os.path.join(fig_dir, filename)
    fig.savefig(path, bbox_inches="tight", dpi=120)
    plt.close(fig)
    print(f"[eval] saved: {path}")


def compute_test_metrics(fitted_models: dict, X_test, y_test) -> pd.DataFrame:
    rows = []
    proba_store = {}
    for name, model in fitted_models.items():
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        proba_store[name] = (y_pred, y_proba)

        rows.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_proba),
        })
    metrics_df = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
    return metrics_df, proba_store


def cross_validate_models(models_builder_fn, X, y, n_folds: int, random_state: int = 42,
                           scoring: str = "f1") -> pd.DataFrame:
    """models_builder_fn() -> dict nama:model (belum fit), dipanggil ulang tiap kali
    supaya tiap model fresh (belum fit) sebelum cross_val_score."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    rows = []
    models = models_builder_fn(random_state=random_state)
    for name, model in models.items():
        print(f"[eval] Cross-validation ({n_folds}-fold, {scoring}) utk {name} ...")
        scores = cross_val_score(model, X, y, cv=skf, scoring=scoring, n_jobs=-1)
        rows.append({
            "Model": name,
            f"CV_{scoring}_mean": scores.mean(),
            f"CV_{scoring}_std": scores.std(),
        })
        print(f"[eval] {name}: {scoring}={scores.mean():.4f} +/- {scores.std():.4f}")
    return pd.DataFrame(rows)


def plot_confusion_matrices(fitted_models: dict, proba_store: dict, y_test, fig_dir: str):
    n = len(fitted_models)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, (name, (y_pred, _)) in enumerate(proba_store.items()):
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i],
                    xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
        axes[i].set_title(name)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    fig.suptitle("Confusion Matrix per Model", y=1.02)
    _save(fig, fig_dir, "20_confusion_matrices.png")


def plot_roc_curves(proba_store: dict, y_test, fig_dir: str):
    fig, ax = plt.subplots(figsize=(7, 6))
    for name, (_, y_proba) in proba_store.items():
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve - Perbandingan Model")
    ax.legend()
    _save(fig, fig_dir, "21_roc_curves.png")


def plot_precision_recall_curves(proba_store: dict, y_test, fig_dir: str):
    fig, ax = plt.subplots(figsize=(7, 6))
    for name, (_, y_proba) in proba_store.items():
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        ax.plot(recall, precision, label=name)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve - Perbandingan Model")
    ax.legend()
    _save(fig, fig_dir, "22_precision_recall_curves.png")


def plot_feature_importance(fitted_models: dict, feature_names, fig_dir: str, top_n=20):
    tree_models = {}
    for name, model in fitted_models.items():
        if hasattr(model, "feature_importances_"):
            tree_models[name] = model.feature_importances_

    if not tree_models:
        return

    ncols = min(3, len(tree_models))
    nrows = int(np.ceil(len(tree_models) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.array(axes).reshape(-1)

    for i, (name, importances) in enumerate(tree_models.items()):
        s = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(top_n)
        sns.barplot(x=s.values, y=s.index, ax=axes[i], color="teal")
        axes[i].set_title(f"Feature Importance - {name}")
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    _save(fig, fig_dir, "23_feature_importance.png")


def plot_metrics_comparison(metrics_df: pd.DataFrame, fig_dir: str):
    melted = metrics_df.melt(id_vars="Model",
                              value_vars=["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"],
                              var_name="Metric", value_name="Score")
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=melted, x="Metric", y="Score", hue="Model", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_title("Perbandingan Metrik Antar Model (Test Set)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    _save(fig, fig_dir, "24_metrics_comparison.png")

## 1. Load semua data

In [11]:
raw_tables = load_all_raw_tables()

[load] application_train: (307511, 122)
[load] application_test: (48744, 121)
[load] bureau: (1716428, 17)
[load] bureau_balance: (27299925, 3)
[load] previous_application: (1670214, 37)
[load] POS_CASH_balance: (10001358, 8)
[load] credit_card_balance: (3840312, 23)
[load] installments_payments: (13605401, 8)


## 2. Join jadi satu dataset besar
Semua tabel anak (bureau, previous_application, POS_CASH_balance,
credit_card_balance, installments_payments) diagregasi ke level
SK_ID_CURR terlebih dahulu (karena 1 SK_ID_CURR bisa punya banyak baris
riwayat), lalu digabung ke application_train / application_test.

In [12]:
df_train_raw, df_test_raw = build_master_dataset(raw_tables)
print("Master train:", df_train_raw.shape)
print("Master test :", df_test_raw.shape)

try:
    df_train_raw.to_parquet(config.MASTER_DATASET_PATH, index=False)
    print(f"Master dataset (train) disimpan ke: {config.MASTER_DATASET_PATH}")
except ImportError:
    df_train_raw.to_csv(config.MASTER_DATASET_CSV_PATH, index=False)
    print(f"(pyarrow/fastparquet tidak tersedia) Master dataset (train) "
          f"disimpan sbg CSV ke: {config.MASTER_DATASET_CSV_PATH}")

[join] Menggabungkan application_train & application_test agar encoding & agregasi konsisten...
[join] application gabungan: (356255, 123)
[join] Agregasi bureau_balance -> bureau ...
[join] bureau_agg: (305811, 229)
[join] Agregasi previous_application ...
[join] prev_agg: (338857, 420)
[join] Agregasi POS_CASH_balance ...
[join] pos_agg: (337252, 52)
[join] Agregasi credit_card_balance ...
[join] cc_agg: (103558, 123)
[join] Agregasi installments_payments ...
[join] ins_agg: (339587, 52)
[join] merge bureau: 123 -> 351 kolom
[join] merge previous_application: 351 -> 770 kolom
[join] merge pos_cash: 770 -> 821 kolom
[join] merge credit_card: 821 -> 943 kolom
[join] merge installments: 943 -> 994 kolom
[join] Dataset akhir (master): (356255, 994)
Master train: (307511, 993)
Master test : (48744, 992)
Master dataset (train) disimpan ke: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\master_dataset.parquet


## 3. Data Wrangling (tahap 1 — sebelum EDA)
- Perbaikan anomali (mis. DAYS_EMPLOYED = 365243)
- Feature engineering rasio finansial (credit/income, annuity/income,
  employed/age, dst — analog debt-to-income & age-to-experience ratio
  pada paper referensi)
- Buang kolom dengan missing value > threshold

Encoding kategorikal SENGAJA ditunda sampai setelah EDA supaya
label kategori (mis. 'M'/'F', 'Higher education') masih terbaca di plot.

In [13]:
df_train, df_test = wrangle_pre_eda(df_train_raw, df_test_raw, config.MISSING_THRESHOLD)
print("Setelah wrangling tahap 1 - train:", df_train.shape, "| test:", df_test.shape)

[wrangling] Membuang 9 kolom dgn missing > 90%
Setelah wrangling tahap 1 - train: (307511, 994) | test: (48744, 993)


## 4. EDA (Exploratory Data Analysis)

In [14]:
corr_target = run_full_eda(df_train, config.FIG_DIR)
if corr_target is not None:
    print("\nTop 15 fitur berkorelasi (absolut) dengan TARGET:")
    print(corr_target.head(15))

[eda] Menjalankan EDA lengkap ...
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\01_target_distribution.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\02_age_distribution.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\03_income_distribution.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\04_income_vs_credit.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\05_credit_amount_distribution.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\06_missing_values.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\07_correlation_he

## 4b. Data Wrangling (tahap 2 — setelah EDA)
- Encoding kategorikal (LabelEncoder utk biner, One-Hot utk multi-kategori)
- Bersihkan nama kolom
- Isi sisa missing value numerik dengan median

In [15]:
df_train, df_test = wrangle_encode(df_train, df_test)
print("Setelah encoding - train:", df_train.shape, "| test:", df_test.shape)

num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in [config.ID_COL, config.TARGET_COL]]
medians = df_train[num_cols].median()
df_train[num_cols] = df_train[num_cols].fillna(medians)
df_test[num_cols] = df_test[[c for c in num_cols if c in df_test.columns]].fillna(medians)

[wrangling] Label-encoded (biner): 4 kolom
[wrangling] One-hot encoded (multi-kategori): 12 kolom
Setelah encoding - train: (307511, 1126) | test: (48744, 1125)


## 5. Feature Selection
Dua tahap sesuai paper: (1) ranking cepat via feature_importances_ model
tree -> ambil top-N kandidat, (2) RFECV pada subsample untuk fine-tune
jumlah & pilihan fitur akhir.

In [16]:
feature_cols = [c for c in df_train.columns if c not in [config.ID_COL, config.TARGET_COL]]
X_full = df_train[feature_cols]
y_full = df_train[config.TARGET_COL].astype(int)

selected_features = run_feature_selection(X_full, y_full, config)
print("\nFitur terpilih untuk modeling:")
print(selected_features)

[feature_selection] SelectFromModel: melatih GradientBoosting utk ranking fitur ...
[feature_selection] 100 fitur teratas dipilih dari 1124 fitur awal.
[feature_selection] RFECV: menjalankan pada subsample n=20000 ...
[feature_selection] RFECV memilih 40 fitur (dari 100 kandidat).
[feature_selection] Fitur final untuk modeling: 40

Fitur terpilih untuk modeling:
['EXT_SOURCE_2', 'EXT_SOURCE_3', 'EXT_SOURCE_1', 'GOODS_PRICE_CREDIT_RATIO', 'CREDIT_TERM', 'INSTAL_PAYMENT_DIFF_MEAN', 'BUREAU_DAYS_CREDIT_ENDDATE_MEAN', 'PREV_NAME_CONTRACT_STATUS_Approved_MEAN', 'PREV_NAME_CONTRACT_STATUS_Refused_MEAN', 'DAYS_EMPLOYED', 'INSTAL_PAYMENT_RATIO_MEAN', 'ANNUITY_INCOME_RATIO', 'PREV_CNT_PAYMENT_STD', 'INSTAL_DAYS_INSTALMENT_STD', 'EMPLOYED_TO_BIRTH_RATIO', 'DAYS_BIRTH', 'INSTAL_DAYS_ENTRY_PAYMENT_STD', 'EMPLOYED_YEARS', 'PREV_DAYS_DECISION_MEAN', 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', 'INSTAL_DAYS_LATE_SUM', 'INSTAL_AMT_PAYMENT_MIN', 'PREV_DAYS_LAST_DUE_1ST_VERSION_SUM', 'BUREAU_AMT_CREDIT_SUM_DEBT_S

## 6. Split Train/Test (stratified 80/20) + Handling Class Imbalance

In [17]:
X = df_train[selected_features]
y = y_full

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, stratify=y, random_state=config.RANDOM_STATE
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Distribusi target train: {y_train.value_counts(normalize=True).to_dict()}")

X_train_smote, y_train_smote = apply_smote(X_train, y_train, config.RANDOM_STATE)

Train: (246008, 40), Test: (61503, 40)
Distribusi target train: {0: 0.9192709180189262, 1: 0.08072908198107379}
[modeling] SMOTE sebelum: {0: 226148, 1: 19860}
[modeling] SMOTE sesudah: {0: 226148, 1: 226148}


## 7. Modeling
Model dilatih pada data hasil SMOTE (di atas train split, TIDAK menyentuh
test set). Semua model juga tetap memakai `class_weight="balanced"`
sebagai lapisan kedua penanganan imbalance (sesuai paper: "dual approach").

Model yang dicoba: **Logistic Regression (wajib)**, Random Forest,
Gradient Boosting, XGBoost, LightGBM, + Voting Ensemble.

In [18]:
fitted_models, timings = train_all_models(
    X_train, y_train, X_train_smote, y_train_smote, config.RANDOM_STATE
)
print("\nWaktu training per model (detik):")
print(pd.Series(timings).sort_values())

[modeling] Melatih LogisticRegression ...
[modeling] LogisticRegression selesai dalam 2.4 detik
[modeling] Melatih RandomForest ...
[modeling] RandomForest selesai dalam 210.6 detik
[modeling] Melatih GradientBoosting ...
[modeling] GradientBoosting selesai dalam 1277.0 detik
[modeling] Melatih XGBoost ...
[modeling] XGBoost selesai dalam 13.4 detik
[modeling] Melatih LightGBM ...
[modeling] LightGBM selesai dalam 12.2 detik
[modeling] Melatih Voting Ensemble (XGBoost + GradientBoosting + RandomForest) ...
[modeling] VotingEnsemble selesai dalam 1430.6 detik

Waktu training per model (detik):
LogisticRegression       2.437054
LightGBM                12.232752
XGBoost                 13.400141
RandomForest           210.560006
GradientBoosting      1276.964347
VotingEnsemble        1430.618725
dtype: float64


## 8. Evaluasi

### 8.1 Metrik pada Test Set

In [19]:
metrics_df, proba_store = compute_test_metrics(fitted_models, X_test, y_test)
print(metrics_df)
metrics_df.to_csv(os.path.join(config.OUTPUT_DIR, "model_metrics_test.csv"), index=False)

                Model  Accuracy  Precision    Recall        F1   ROC_AUC
0  LogisticRegression  0.695592   0.162835  0.669084  0.261925  0.746690
1        RandomForest  0.797148   0.167508  0.381067  0.232718  0.691306
2      VotingEnsemble  0.852901   0.196189  0.265458  0.225627  0.705517
3    GradientBoosting  0.825992   0.175031  0.311178  0.224043  0.694920
4             XGBoost  0.883518   0.212849  0.164149  0.185354  0.709513
5            LightGBM  0.902769   0.231907  0.088419  0.128026  0.717675


### 8.2 Cross-Validation (Stratified 5-fold, F1) — model belum-fit (fresh)
Dijalankan pada data train (non-SMOTE) memakai class_weight='balanced'
supaya representatif sebagaimana skenario deployment nyata.

In [20]:
cv_df = cross_validate_models(
    build_models, X_train, y_train, n_folds=config.N_CV_FOLDS,
    random_state=config.RANDOM_STATE, scoring="f1"
)
print(cv_df)
cv_df.to_csv(os.path.join(config.OUTPUT_DIR, "model_cv_scores.csv"), index=False)

[eval] Cross-validation (5-fold, f1) utk LogisticRegression ...
[eval] LogisticRegression: f1=0.2595 +/- 0.0015
[eval] Cross-validation (5-fold, f1) utk RandomForest ...
[eval] RandomForest: f1=0.2809 +/- 0.0025
[eval] Cross-validation (5-fold, f1) utk GradientBoosting ...
[eval] GradientBoosting: f1=0.0386 +/- 0.0028
[eval] Cross-validation (5-fold, f1) utk XGBoost ...
[eval] XGBoost: f1=0.0543 +/- 0.0036
[eval] Cross-validation (5-fold, f1) utk LightGBM ...
[eval] LightGBM: f1=0.2831 +/- 0.0020
                Model  CV_f1_mean  CV_f1_std
0  LogisticRegression    0.259509   0.001546
1        RandomForest    0.280920   0.002533
2    GradientBoosting    0.038619   0.002841
3             XGBoost    0.054320   0.003608
4            LightGBM    0.283061   0.001977


### 8.3 Visualisasi: Confusion Matrix, ROC Curve, PR Curve, Feature Importance

In [21]:
plot_confusion_matrices(fitted_models, proba_store, y_test, config.FIG_DIR)
plot_roc_curves(proba_store, y_test, config.FIG_DIR)
plot_precision_recall_curves(proba_store, y_test, config.FIG_DIR)
plot_feature_importance(fitted_models, selected_features, config.FIG_DIR)
plot_metrics_comparison(metrics_df, config.FIG_DIR)

[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\20_confusion_matrices.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\21_roc_curves.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\22_precision_recall_curves.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\23_feature_importance.png
[eval] saved: c:\Users\Lenovo ThinkPad T480\Documents\Rakamin Academy\Week 5\home_credit_project\outputs\figures\24_metrics_comparison.png


## 9. Ringkasan
Model terbaik berdasarkan F1-score dan ROC AUC dicetak di bawah. Semua
metrik, plot, dan model tersimpan di folder `outputs/`.

Hyperparameter tuning (GridSearchCV/RandomizedSearchCV/Bayesian
Optimization) menjadi langkah berikutnya.

In [22]:
best_by_f1 = metrics_df.sort_values("F1", ascending=False).iloc[0]
best_by_auc = metrics_df.sort_values("ROC_AUC", ascending=False).iloc[0]
print(f"Model terbaik (F1)     : {best_by_f1['Model']} -> F1={best_by_f1['F1']:.4f}")
print(f"Model terbaik (ROC AUC): {best_by_auc['Model']} -> ROC_AUC={best_by_auc['ROC_AUC']:.4f}")

print("\nSelesai. Cek folder outputs/ untuk semua hasil (figures, csv metrik, dataset gabungan).")

Model terbaik (F1)     : LogisticRegression -> F1=0.2619
Model terbaik (ROC AUC): LogisticRegression -> ROC_AUC=0.7467

Selesai. Cek folder outputs/ untuk semua hasil (figures, csv metrik, dataset gabungan).
